# Week 7 Assignment — Feature Engineering
### Name: Yasaa Usman Ahmad Khan

**Submit as:** this notebook only -- no separate report this week

**This continues your Week 6 assignment.** You will reuse your own Week 6 raw-feature clustering and PCA, then ask the three questions Lab 7 asked of the synthetic loan data — this time on real data where nobody designed the answer in advance:

1. Do engineered features give **sharper clusters** than raw features?
2. Does PCA now show a **risk-relevant** structure instead of a size axis?


## The dataset

**Default of Credit Card Clients** — 30,000 real credit card customers in Taiwan, 2005. UCI Machine Learning Repository, donated by I-Cheng Yeh. CC BY 4.0.
Source: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

**Citation (include in your report):** Yeh, I. (2009). *Default of Credit Card Clients* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C55S3H

| Column(s) | Meaning |
|---|---|
| `LIMIT_BAL` | Credit limit (NT dollars) |
| `SEX`, `EDUCATION`, `MARRIAGE`, `AGE` | Demographics |
| `PAY_0`, `PAY_2` … `PAY_6` | Repayment status. **`PAY_0` = September (most recent), `PAY_6` = April (oldest)** |
| `BILL_AMT1` … `BILL_AMT6` | Bill statement amount. **`BILL_AMT1` = September, `BILL_AMT6` = April** |
| `PAY_AMT1` … `PAY_AMT6` | Amount paid in that month. **`PAY_AMT1` = September, `PAY_AMT6` = April** |
| `default` | **Target:** defaulted the following month (1) or not (0) |

> **Read the time order carefully.** The numbering runs *backwards* in time (1 = most recent). A payment made in a month goes toward the bill issued the month *before* — think about which `BILL_AMT` a given `PAY_AMT` actually pays off.

## Setup — load the data and bring your Week 6 work forward

In [1]:
# pip install ucimlrepo   (uncomment if needed)
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

dataset = fetch_ucirepo(id=350)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

# The repository has used two naming schemes for this dataset (X1..X23 / Y, or descriptive names).
# Normalise to the descriptive names used in this course so everyone's code matches.
RENAME = {"X1": "LIMIT_BAL", "X2": "SEX", "X3": "EDUCATION", "X4": "MARRIAGE", "X5": "AGE",
          "X6": "PAY_0", "X7": "PAY_2", "X8": "PAY_3", "X9": "PAY_4", "X10": "PAY_5", "X11": "PAY_6",
          **{f"X{11 + i}": f"BILL_AMT{i}" for i in range(1, 7)},
          **{f"X{17 + i}": f"PAY_AMT{i}" for i in range(1, 7)},
          "Y": "default", "default.payment.next.month": "default", "PAY_1": "PAY_0"}
df = df.rename(columns={k: v for k, v in RENAME.items() if k in df.columns})

needed = ["LIMIT_BAL", "AGE", "PAY_0", "BILL_AMT1", "PAY_AMT1", "default"]
missing = [c for c in needed if c not in df.columns]
assert not missing, f"Unexpected column names, missing {missing}. Found: {list(df.columns)}"

print("Shape:", df.shape, "| default rate:", round(df["default"].mean(), 3))
df.head(3)

Shape: (30000, 24) | default rate: 0.221


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


### Bring your Week 6 raw-feature work forward

Paste your own Week 6 code (column classification, scaling, K-Means, PCA) in the cell below. When it runs, the notebook must define these **four names** so the comparisons later in this assignment work:

* `RAW_FEATURES` — the list of columns you clustered on
* `K` — the number of clusters you chose in Week 6 (keep it for a fair comparison)
* `raw_labels` — the cluster labels from your Week 6 K-Means
* `raw_pca_pc1_variance` — the share of variance PC1 explained in your Week 6 PCA

In [2]:
# Week 6 raw-feature work, brought forward (same logic and same random_state as my Week 6 notebook)
MONEY = ["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1, 7)] + [f"PAY_AMT{i}" for i in range(1, 7)]
STATUS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

RAW_FEATURES = MONEY + ["AGE"] + STATUS     # 13 size-scaled money columns + 7 bounded columns
K = 3                                       # elbow choice from Week 6

X_raw_scaled = StandardScaler().fit_transform(df[RAW_FEATURES])
raw_labels = KMeans(n_clusters=K, random_state=42, n_init=10).fit(X_raw_scaled).labels_

raw_pca = PCA().fit(X_raw_scaled)
raw_pca_pc1_variance = raw_pca.explained_variance_ratio_[0]

In [3]:
# Bring-forward check (provided): run this after pasting. It should print your Week 6 raw result.
print("features:", len(RAW_FEATURES), "| K =", K, "| PC1 variance:", round(float(raw_pca_pc1_variance), 3))
print("raw cluster sizes:", np.bincount(raw_labels).tolist())
print("raw default rates:", [round(float(df["default"][raw_labels == k].mean()), 3) for k in range(K)])

features: 20 | K = 3 | PC1 variance: 0.327
raw cluster sizes: [15924, 10519, 3557]
raw default rates: [0.266, 0.165, 0.189]


**Reference (Lab 7, synthetic loan data):** raw K-Means gave default rates of `0.271 / 0.487 / 0.333` (blurry); engineered K-Means gave `0.757 / 0.294 / 0.295` (one sharp high-risk segment).
Your real-data numbers will differ — that is expected, and *how* they differ is part of what you are asked to explain.

---
## Part 1 — Diagnose before you transform

In Lab 7, the textbook fix (log-transform income) turned out to make things **worse**, because income was already symmetric. Do not assume here either.

1. For every money-amount column (`LIMIT_BAL`, `BILL_AMT1–6`, `PAY_AMT1–6`), compute the **skew**, and the skew after any transform you are considering. Show it in a small table.
2. Look for problems a model would trip over: **negative or zero values** in columns you might divide by, extreme outliers, and **category codes that do not match the documentation** (check `.value_counts()` for `EDUCATION`, `MARRIAGE` and the `PAY_*` columns against the table above — confirm for yourself rather than trusting the documentation).
3. Decide, column by column, whether to **log**, **winsorize**, **ratio** or **leave alone**.

In [4]:
BILLS = [f"BILL_AMT{i}" for i in range(1, 7)]
PAYAMT = [f"PAY_AMT{i}" for i in range(1, 7)]

# 1. Skew of every money column, raw and after the transforms I am considering
rows = []
for c in MONEY:
    x = df[c]
    rows.append({
        "column": c,
        "skew raw": skew(x),
        "skew log (positive rows only)": skew(np.log(x[x > 0])),
        "skew signed log (all rows)": skew(np.sign(x) * np.log1p(np.abs(x))),
        "rows zero or negative": int((x <= 0).sum()),
        "share zero or negative": (x <= 0).mean(),
    })
skew_table = pd.DataFrame(rows).set_index("column")
display(skew_table.round(2))

,skew raw,skew log (positive rows only),skew signed log (all rows),rows zero or negative,share zero or negative
column,,,,,
LIMIT_BAL,0.99,-0.51,-0.51,0,0.00
BILL_AMT1,2.66,-0.73,-2.08,2598,0.09
BILL_AMT2,2.71,-0.78,-1.94,3175,0.11
BILL_AMT3,3.09,-0.80,-1.85,3525,0.12
BILL_AMT4,2.82,-0.77,-1.78,3870,0.13
BILL_AMT5,2.88,-0.77,-1.68,4161,0.14
BILL_AMT6,2.85,-0.76,-1.56,4708,0.16
PAY_AMT1,14.67,-0.68,-1.29,5249,0.17
PAY_AMT2,30.45,-0.66,-1.24,5396,0.18


In [5]:
# 2. Problems a model would trip over
print("Zero or negative values in columns I might divide by:")
print("  LIMIT_BAL smallest value:", int(df["LIMIT_BAL"].min()), "(safe denominator)")
print("  BILL_AMT1-6 negative rows:", {c: int((df[c] < 0).sum()) for c in BILLS})
print("  BILL_AMT1-6 zero rows:    ", {c: int((df[c] == 0).sum()) for c in BILLS})

print("\nExtreme outliers (max versus median):")
for c in ["BILL_AMT3", "PAY_AMT2", "LIMIT_BAL"]:
    print(f"  {c}: median {df[c].median():,.0f} | 99th percentile {df[c].quantile(.99):,.0f} | max {df[c].max():,.0f}")

print("\nCategory codes against the documentation:")
for c in ["EDUCATION", "MARRIAGE"]:
    print(f"  {c}:", df[c].value_counts().sort_index().to_dict())
print(pd.concat({c: df[c].value_counts().sort_index() for c in STATUS}, axis=1).fillna(0).astype(int).to_string())

Zero or negative values in columns I might divide by:
  LIMIT_BAL smallest value: 10000 (safe denominator)
  BILL_AMT1-6 negative rows: {'BILL_AMT1': 590, 'BILL_AMT2': 669, 'BILL_AMT3': 655, 'BILL_AMT4': 675, 'BILL_AMT5': 655, 'BILL_AMT6': 688}
  BILL_AMT1-6 zero rows:     {'BILL_AMT1': 2008, 'BILL_AMT2': 2506, 'BILL_AMT3': 2870, 'BILL_AMT4': 3195, 'BILL_AMT5': 3506, 'BILL_AMT6': 4020}

Extreme outliers (max versus median):
  BILL_AMT3: median 20,088 | 99th percentile 325,030 | max 1,664,089
  PAY_AMT2: median 2,009 | 99th percentile 76,651 | max 1,684,259
  LIMIT_BAL: median 140,000 | 99th percentile 500,000 | max 1,000,000

Category codes against the documentation:
  EDUCATION: {0: 14, 1: 10585, 2: 14030, 3: 4917, 4: 123, 5: 280, 6: 51}
  MARRIAGE: {0: 54, 1: 13659, 2: 15964, 3: 323}
    PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6
-2   2759   3782   4085   4348   4546   4895
-1   5686   6050   5938   5687   5539   5740
 0  14737  15730  15764  16455  16947  16286
 1   3688     28      4

In [6]:
# 3. Decision, column by column (evidence is read from the table above, not typed in)
st = skew_table
decisions = pd.DataFrame([
    {"column(s)": "LIMIT_BAL", "decision": "log (as a feature)",
     "evidence": f"skew {st.loc['LIMIT_BAL','skew raw']:.2f} -> {st.loc['LIMIT_BAL','skew log (positive rows only)']:.2f}, no zero or negative rows"},
    {"column(s)": "BILL_AMT1-6", "decision": "ratio (over LIMIT_BAL); do not log",
     "evidence": f"{st.loc[BILLS,'rows zero or negative'].min():,} to {st.loc[BILLS,'rows zero or negative'].max():,} rows are zero or negative; signed log leaves skew {st.loc[BILLS,'skew signed log (all rows)'].min():.2f} to {st.loc[BILLS,'skew signed log (all rows)'].max():.2f}"},
    {"column(s)": "PAY_AMT1-6", "decision": "ratio (over what was owed); do not log",
     "evidence": f"raw skew {st.loc[PAYAMT,'skew raw'].min():.1f} to {st.loc[PAYAMT,'skew raw'].max():.1f}; log flips it to {st.loc[PAYAMT,'skew signed log (all rows)'].min():.2f} to {st.loc[PAYAMT,'skew signed log (all rows)'].max():.2f} because {st.loc[PAYAMT,'share zero or negative'].min():.0%} to {st.loc[PAYAMT,'share zero or negative'].max():.0%} of rows are zero"},
    {"column(s)": "engineered ratios", "decision": "winsorize + flag", "evidence": "see Part 2"},
    {"column(s)": "AGE, PAY_* statuses", "decision": "leave alone", "evidence": f"AGE skew {skew(df['AGE']):.2f}; statuses are small fixed codes"},
]).set_index("column(s)")
pd.set_option("display.max_colwidth", 200)
display(decisions)

,decision,evidence
column(s),,
LIMIT_BAL,log (as a feature),"skew 0.99 -> -0.51, no zero or negative rows"
BILL_AMT1-6,ratio (over LIMIT_BAL); do not log,"2,598 to 4,708 rows are zero or negative; signed log leaves skew -2.08 to -1.56"
PAY_AMT1-6,ratio (over what was owed); do not log,raw skew 10.6 to 30.5; log flips it to -1.29 to -0.85 because 17% to 24% of rows are zero
engineered ratios,winsorize + flag,see Part 2
"AGE, PAY_* statuses",leave alone,AGE skew 0.73; statuses are small fixed codes


**Write-up (Part 1):** Which transforms did you decide on, and what evidence (numbers from your table) supports each? Name at least one case where a transform you considered was **not** worth it.

**LIMIT_BAL: log, used as a feature.** Skew goes from 0.99 to -0.51, and there are no zero or negative rows (the smallest limit is 10,000), so the log is always defined. It is a modest gain, but a real one.

**BILL_AMT1 to 6: do not log; use as the top of ratios over `LIMIT_BAL` instead.** Between 2,598 and 4,708 rows (9% to 16%) are zero or negative, so a plain log is undefined for them. The signed log keeps every row but leaves skew at -2.08 to -1.56: the pile of zeros turns into a long left tail. **This is the transform that was not worth it.**

**PAY_AMT1 to 6: do not log; use inside a payment ratio.** Raw skew is extreme (10.64 to 30.45), but the log flips it to -1.29 to -0.85 because 17% to 24% of the values are exactly zero, and a spike at zero cannot be fixed by a log.

**Engineered ratios: winsorize and flag** (Part 2). **AGE and the `PAY_*` statuses: leave alone** (AGE skew is 0.73; the statuses are small fixed codes).

**Other problems found.** `EDUCATION` has undocumented codes 0, 5, 6 (345 rows) and `MARRIAGE` has code 0 (54 rows); I do not use `SEX`, `EDUCATION` or `MARRIAGE` in my features, so these do not affect my results. The `PAY_*` columns contain -2 and 0, which are not in the documentation, so I treat -2, -1 and 0 all as "not late" and count only status 1 or more as late. There are extreme outliers (`BILL_AMT3`: median 20,088, max 1,664,089; `PAY_AMT2`: median 2,009, 99th percentile 76,651, max 1,684,259).

The contrast with Lab 7 is useful: there the log made income worse because income was already symmetric. Here the log helps the one column with no zeros (`LIMIT_BAL`) and fails on the columns that contain zeros and negatives.

---
## Part 2 — Engineer the features 

Design **at least four** engineered features from the raw columns, including **at least one ratio**. Ideas (use any, change any, invent your own):

* **Utilization:** how much of the credit limit is in use (bill relative to limit)
* **Payment ratio:** how much of a bill is actually paid off (mind which bill a payment applies to — and mind bills that are zero or negative)
* **Trend over the six months:** is repayment status or bill size improving, worsening or flat?
* **Persistence:** in how many of the six months was the customer late?

**Requirements (all graded):**

1. Every ratio must **handle bad denominators** (zero or negative values) deliberately — state what you did and why. Do not let `inf`/`NaN` leak into the model silently.
2. **Winsorize** the skewed engineered features at a percentile you justify, **and create an `extreme_flag`** (computed from the *uncapped* values, *before* clipping).
3. Package everything in one function:

```python
def add_features(data, caps=None):
    ...
    return new_df, caps
```

   It must **learn** its caps only when `caps is None` and **reuse** them when `caps` is passed in. You will rely on this in Part 4.

In [7]:
LATE_ORDER = ["PAY_6", "PAY_5", "PAY_4", "PAY_3", "PAY_2", "PAY_0"]   # oldest month (April) to newest (September)
WINSORIZE = ["utilization", "payment_ratio", "util_change"]

def safe_ratio(num, den, fallback):
    """num / den, but wherever den is zero or negative (or the result is not finite) use `fallback`."""
    den = pd.Series(den, index=num.index)
    out = num / den.where(den > 0)
    return out.fillna(fallback).replace([np.inf, -np.inf], fallback)

def add_features(data, caps=None, q=0.99):
    d = data.copy()

    # (1) ratios. Negative bills are credit balances (customer overpaid), so they count as zero use.
    d["utilization"] = safe_ratio(d["BILL_AMT1"].clip(lower=0), d["LIMIT_BAL"], 0.0)
    # PAY_AMTk pays off the bill issued the month before, which is BILL_AMT(k+1): PAY_AMT1..5 pay BILL_AMT2..6.
    owed = d[BILLS[1:]].clip(lower=0).sum(axis=1)
    d["payment_ratio"] = safe_ratio(d[PAYAMT[:5]].sum(axis=1), owed, 1.0)     # nothing owed -> nothing missed -> 1.0
    d["util_change"] = safe_ratio(d["BILL_AMT1"].clip(lower=0) - d["BILL_AMT6"].clip(lower=0), d["LIMIT_BAL"], 0.0)

    # persistence and trend, from the repayment statuses (only 1 or more months late counts as late)
    d["late_months"] = (d[STATUS] >= 1).sum(axis=1)
    t = np.arange(6) - 2.5                                                    # least-squares slope over 6 months
    d["delay_trend"] = d[LATE_ORDER].clip(lower=0).to_numpy() @ (t / (t ** 2).sum())

    d["log_limit"] = np.log(d["LIMIT_BAL"])

    # (2) learn caps ONLY if none were given
    if caps is None:
        caps = {c: d[c].quantile(q) for c in WINSORIZE}

    # (3) flag from the UNCAPPED values, then clip
    d["extreme_flag"] = 0
    for c in WINSORIZE:
        d["extreme_flag"] |= (d[c] > caps[c]).astype(int)
        d[c] = d[c].clip(upper=caps[c])
    return d, caps

feat, caps = add_features(df)
ENG_NEW = ["utilization", "payment_ratio", "util_change", "late_months", "delay_trend", "log_limit"]
print("shape:", feat.shape)
print("caps learned at the 99th percentile:", {c: round(float(v), 3) for c, v in caps.items()})
print("NaN or inf left in engineered columns:", int(feat[ENG_NEW].replace([np.inf, -np.inf], np.nan).isna().sum().sum()))
print("flagged rows:", int(feat["extreme_flag"].sum()), f"({feat['extreme_flag'].mean():.1%})")
print(f"default rate flagged: {df['default'][feat['extreme_flag'] == 1].mean():.3f} | not flagged: {df['default'][feat['extreme_flag'] == 0].mean():.3f}")

shape: (30000, 31)
caps learned at the 99th percentile: {'utilization': 1.232, 'payment_ratio': 1.333, 'util_change': 1.001}
NaN or inf left in engineered columns: 0
flagged rows: 748 (2.5%)
default rate flagged: 0.241 | not flagged: 0.221


In [8]:
# Why the 99th percentile? Compare it with the 95th used in the lab.
rows = []
for c in WINSORIZE:
    raw_x, _ = add_features(df, caps={k: np.inf for k in WINSORIZE})
    x = raw_x[c]
    row = {"feature": c, "skew uncapped": skew(x)}
    for q in (0.95, 0.99):
        cap = x.quantile(q)
        row[f"cap p{int(q*100)}"] = cap
        row[f"skew p{int(q*100)}"] = skew(x.clip(upper=cap))
        row[f"rows above p{int(q*100)}"] = int((x > cap).sum())
    rows.append(row)
display(pd.DataFrame(rows).set_index("feature").round(2))

# Rows where payment_ratio had to use the fallback (nothing owed in months 2-6)
owed = df[BILLS[1:]].clip(lower=0).sum(axis=1)
print("\nRows with nothing owed in months 2-6 (payment_ratio fallback = 1.0):", int((owed <= 0).sum()),
      f"| their default rate {df['default'][owed <= 0].mean():.3f} vs overall {df['default'].mean():.3f}")
print("Rows whose bill exceeds the credit limit (utilization above 1):", int((df['BILL_AMT1'] > df['LIMIT_BAL']).sum()))

# Why is payment_ratio so extreme? Look at the amount owed behind the huge ratios.
raw_feat, _ = add_features(df, caps={k: np.inf for k in WINSORIZE})
huge = (raw_feat["payment_ratio"] > 5).to_numpy()
print(f"\nRows with payment_ratio above 5: {int(huge.sum())} | median amount owed behind them: {owed[huge].median():,.0f} "
      f"vs {owed[owed > 0].median():,.0f} for everyone who owed something")

,skew uncapped,cap p95,skew p95,rows above p95,cap p99,skew p99,rows above p99
feature,,,,,,,
utilization,0.82,1.01,0.33,1500,1.23,0.37,300
payment_ratio,92.13,1.01,0.64,1500,1.33,0.69,294
util_change,1.66,0.72,0.53,1500,1.00,0.94,300



Rows with nothing owed in months 2-6 (payment_ratio fallback = 1.0): 1406 | their default rate 0.312 vs overall 0.221
Rows whose bill exceeds the credit limit (utilization above 1): 2115

Rows with payment_ratio above 5: 37 | median amount owed behind them: 780 vs 111,922 for everyone who owed something


In [9]:
# The function must reuse caps it is given (what a test set or production data has to do)
train_part, test_part = df.iloc[:20000], df.iloc[20000:]
_, caps_train = add_features(train_part)
test_feat, caps_back = add_features(test_part, caps=caps_train)
print("caps returned unchanged when passed in:", caps_back == caps_train)
print("flagged share on the held-out part, using the TRAIN caps:", f"{test_feat['extreme_flag'].mean():.1%}")

# Quick sanity check of each engineered feature against the label (a diagnostic only; I did not pick features with it)
print("\nSpearman correlation with default:")
print(feat[ENG_NEW].corrwith(df["default"], method="spearman").round(3).to_dict())

caps returned unchanged when passed in: True
flagged share on the held-out part, using the TRAIN caps: 2.5%

Spearman correlation with default:
{'utilization': 0.073, 'payment_ratio': -0.132, 'util_change': -0.08, 'late_months': 0.388, 'delay_trend': 0.12, 'log_limit': -0.17}


**Write-up (Part 2):** For **each** engineered feature, one or two sentences of *domain* reasoning — why would a risk analyst expect it to matter? Then one sentence on how many rows your `extreme_flag` marked and whether those rows default more or less than the rest.

**Bad denominators.** `LIMIT_BAL` is never zero (minimum 10,000). Negative bills are credit balances, so I count them as zero use. For `payment_ratio`, if a customer owed nothing in months 2 to 6 (1,406 customers), I set the ratio to 1.0 (nothing owed, nothing missed) rather than leaving a NaN or inf. One caution: those customers default at 0.312 versus 0.221 overall, so calling them perfect payers is imperfect. A `safe_ratio` helper does this for every ratio, and I checked that no NaN or inf is left (0).

**The features, and why an analyst would expect each to matter:**

1. **`utilization`** (latest bill over credit limit): shows how much of their limit a customer is using. Someone near their limit has little room for a shock, and the same bill means different things at different limits (2,115 customers are over their limit).
2. **`payment_ratio`** (payments over the bills they pay off): shows whether a customer actually pays down what they owe or only carries balances. `PAY_AMTk` pays the bill issued the month before, so `PAY_AMT1` to `PAY_AMT5` are compared with `BILL_AMT2` to `BILL_AMT6`. The raw ratio reaches 376 because the 37 customers with a ratio above 5 owed a median of only 780 (against 111,922 for everyone else), which is why it needs winsorizing.
3. **`util_change`** (change in utilization from April to September): a trend in bill size. A customer whose balance is growing relative to their limit is getting into more trouble than one who is flat.
4. **`late_months`** (months late out of six): persistence. Late once is different from late repeatedly, and repeated lateness is the classic warning sign for default. It has the strongest rank correlation with default of my features (0.388).
5. **`delay_trend`** (slope of months-late over the six months): direction. Two customers with the same current status are very different if one is improving and the other is getting worse.
6. **`log_limit`** (log of credit limit): the limit is the lender's own estimate of what the customer can handle, and the log tames its skew.

One caveat on lateness: `PAY_0` looks coded differently from `PAY_2` to `PAY_6` (status 1 appears 3,688 times in `PAY_0` but at most 28 times elsewhere), so September lateness may be counted more generously than earlier months.

**Winsorizing.** I cap `utilization`, `payment_ratio` and `util_change` at the **99th percentile** (caps 1.232, 1.333, 1.001). Skew drops from 0.82, 92.13 and 1.66 to 0.37, 0.69 and 0.94. The 95th percentile (used in the lab) would clip 1,500 customers per feature and set the `payment_ratio` cap at 1.01, which is just "paid in full", so it would call ordinary full payers extreme. The 99th clips only about 300 per feature, the real tail. I did not winsorize `late_months` or `delay_trend` because they are bounded and have no outliers to cap.

**`extreme_flag`** is computed from the uncapped values before clipping. It marked 748 rows (2.5%), and those rows default slightly more than the rest (0.241 vs 0.221).

---
## Part 3 — Re-cluster and re-run PCA

Using your engineered features (**same `K` as Week 6**, unless you can justify a change):

1. Scale, fit K-Means, and **profile** each cluster: size, **default rate**, and the means of your key features. Compare directly with your raw result (side-by-side table).
2. Run PCA on the engineered features. Report the variance table and a **loadings table for PC1 and PC2**. Did PC1 change what it represents?
3. **Failure case (as in Lab 7, Task 2.3):** re-run once **without winsorizing**. Report what happened to the cluster sizes. Does one tiny cluster appear? Who is in it?
4. **Do the clusters separate on risk — or on something else?** Check each pair of clusters: do clusters that differ strongly on a feature also differ in default rate?

Keep the flag and any binned/categorical columns **out of K-Means** (as in the lab), and say whether you did.

In [10]:
ENG = ["utilization", "payment_ratio", "late_months", "delay_trend", "util_change", "log_limit", "AGE"]
# extreme_flag is NOT in ENG, and I created no binned or category columns, so nothing 0/1 or categorical enters K-Means.

X_eng = StandardScaler().fit_transform(feat[ENG])
eng_labels = KMeans(n_clusters=K, random_state=42, n_init=10).fit(X_eng).labels_

def rank_view(labels):
    """cluster summary ordered from highest to lowest default rate"""
    t = (df.assign(cluster=labels).groupby("cluster")["default"].agg(size="size", default_rate="mean"))
    return t.sort_values("default_rate", ascending=False).reset_index(drop=True)

rv_raw, rv_eng = rank_view(raw_labels), rank_view(eng_labels)
side = pd.concat({"raw": rv_raw, "engineered": rv_eng}, axis=1)
side.index = ["riskiest cluster", "middle cluster", "safest cluster"]
display(side.round(3))

base = df["default"].mean()
for name, rv in (("raw", rv_raw), ("engineered", rv_eng)):
    top = rv.iloc[0]
    captured = top["size"] * top["default_rate"] / df["default"].sum()
    print(f"{name:10s} riskiest cluster: {top['size']/len(df):.1%} of customers, default rate {top['default_rate']:.3f} "
          f"(lift {top['default_rate']/base:.2f}x), holds {captured:.1%} of all defaulters | spread of rates {rv['default_rate'].max()-rv['default_rate'].min():.3f}")

print("\nEngineered cluster means (original units):")
display(feat.assign(cluster=eng_labels, default=df["default"]).groupby("cluster")[ENG + ["default"]].mean().round(2).T)

raw              engineered             
                   size default_rate       size default_rate
riskiest cluster  15924        0.266       4293        0.592
middle cluster     3557        0.189      12449        0.165
safest cluster    10519        0.165      13258        0.154

raw        riskiest cluster: 53.1% of customers, default rate 0.266 (lift 1.20x), holds 63.8% of all defaulters | spread of rates 0.101
engineered riskiest cluster: 14.3% of customers, default rate 0.592 (lift 2.68x), holds 38.3% of all defaulters | spread of rates 0.438

Engineered cluster means (original units):


cluster,0,1,2
utilization,0.73,0.64,0.06
payment_ratio,0.10,0.08,0.77
late_months,0.34,3.88,0.31
delay_trend,-0.03,0.21,0.02
util_change,0.25,0.06,-0.03
log_limit,11.38,11.02,12.13
AGE,35.00,35.14,36.05
default,0.16,0.59,0.15


In [11]:
eng_pca = PCA().fit(X_eng)
eng_evr = eng_pca.explained_variance_ratio_
Z_eng = eng_pca.transform(X_eng)

var_table = pd.DataFrame({"raw: explained": raw_pca.explained_variance_ratio_[:7]}, index=[f"PC{i}" for i in range(1, 8)])
var_table["engineered: explained"] = eng_evr
var_table["engineered: cumulative"] = eng_evr.cumsum()
display(var_table.round(3))

print("Engineered PCA loadings (PC1 and PC2):")
eng_load = pd.DataFrame(eng_pca.components_[:2].T, index=ENG, columns=["PC1", "PC2"])
display(eng_load.round(3))

print("Raw PCA (Week 6) for comparison, largest PC1 and PC2 loadings:")
for i in range(2):
    ld = pd.Series(raw_pca.components_[i], index=RAW_FEATURES)
    print(f"  raw PC{i+1}:", ld.sort_values(key=abs, ascending=False).head(4).round(2).to_dict())

raw_Z = raw_pca.transform(X_raw_scaled)
print("\nCorrelation of the PC scores with default:")
print("  raw       PC1 %.3f | PC2 %.3f" % (np.corrcoef(raw_Z[:, 0], df["default"])[0, 1], np.corrcoef(raw_Z[:, 1], df["default"])[0, 1]))
print("  engineered PC1 %.3f | PC2 %.3f" % (np.corrcoef(Z_eng[:, 0], df["default"])[0, 1], np.corrcoef(Z_eng[:, 1], df["default"])[0, 1]))

,raw: explained,engineered: explained,engineered: cumulative
PC1,0.327,0.329,0.329
PC2,0.203,0.177,0.506
PC3,0.075,0.150,0.656
PC4,0.050,0.139,0.795
PC5,0.046,0.100,0.894
PC6,0.044,0.076,0.971
PC7,0.044,0.029,1.000


Engineered PCA loadings (PC1 and PC2):


,PC1,PC2
utilization,0.597,-0.189
payment_ratio,-0.535,-0.045
late_months,0.230,0.657
delay_trend,0.098,0.028
util_change,0.365,-0.594
log_limit,-0.393,-0.334
AGE,-0.080,-0.258


Raw PCA (Week 6) for comparison, largest PC1 and PC2 loadings:
  raw PC1: {'BILL_AMT4': 0.35, 'BILL_AMT5': 0.35, 'BILL_AMT3': 0.35, 'BILL_AMT2': 0.35}
  raw PC2: {'PAY_4': 0.35, 'PAY_3': 0.35, 'PAY_2': 0.34, 'PAY_5': 0.34}

Correlation of the PC scores with default:
  raw       PC1 0.083 | PC2 0.286
  engineered PC1 0.179 | PC2 0.290


In [12]:
# Failure case (Lab 7, Task 2.3): same pipeline, but caps at the 100th percentile = nothing is clipped
nocap, _ = add_features(df, q=1.0)
nocap_labels = KMeans(n_clusters=K, random_state=42, n_init=10).fit(StandardScaler().fit_transform(nocap[ENG])).labels_

print("cluster sizes WITH winsorizing:   ", np.bincount(eng_labels).tolist())
print("cluster sizes WITHOUT winsorizing:", np.bincount(nocap_labels).tolist())
print("default rates without winsorizing:", [round(float(df['default'][nocap_labels == k].mean()), 3) for k in range(K)])

small = int(np.argmin(np.bincount(nocap_labels)))
print(f"\nSmallest cluster without winsorizing: cluster {small}, {int((nocap_labels == small).sum())} customers; "
      f"mean late_months {nocap['late_months'][nocap_labels == small].mean():.2f} (all customers: {nocap['late_months'].mean():.2f})")

pr_c, pr_n = feat["payment_ratio"], nocap["payment_ratio"]
print(f"\npayment_ratio: max {pr_n.max():.0f} uncapped vs {pr_c.max():.2f} capped | std {pr_n.std():.2f} uncapped vs {pr_c.std():.2f} capped")
print("customers with payment_ratio above 5 (uncapped):", int((pr_n > 5).sum()),
      "| which cluster they land in:", np.bincount(nocap_labels[(pr_n > 5).to_numpy()], minlength=K).tolist())

print("\nCluster means WITHOUT winsorizing (original units):")
display(nocap.assign(cluster=nocap_labels, default=df["default"]).groupby("cluster")[ENG + ["default"]].mean().round(2).T)

cluster sizes WITH winsorizing:    [12449, 4293, 13258]
cluster sizes WITHOUT winsorizing: [8914, 16845, 4241]
default rates without winsorizing: [0.181, 0.149, 0.591]

Smallest cluster without winsorizing: cluster 2, 4241 customers; mean late_months 3.96 (all customers: 0.83)

payment_ratio: max 376 uncapped vs 1.33 capped | std 2.90 uncapped vs 0.42 capped
customers with payment_ratio above 5 (uncapped): 37 | which cluster they land in: [6, 30, 1]

Cluster means WITHOUT winsorizing (original units):


cluster,0,1,2
utilization,0.87,0.14,0.63
payment_ratio,0.13,0.70,0.11
late_months,0.27,0.34,3.96
delay_trend,-0.00,0.00,0.19
util_change,0.37,-0.02,0.03
log_limit,11.21,12.07,10.98
AGE,34.89,35.93,34.98
default,0.18,0.15,0.59


In [13]:
# Do clusters separate on risk or on something else? Compare every pair of clusters.
from itertools import combinations
cl_means = feat.assign(cluster=eng_labels).groupby("cluster")[ENG].mean()
overall_sd = feat[ENG].std()
rate = df.assign(cluster=eng_labels).groupby("cluster")["default"].mean()
sizes = np.bincount(eng_labels)

rows = []
for a, b in combinations(range(K), 2):
    gap = ((cl_means.loc[a] - cl_means.loc[b]).abs() / overall_sd).sort_values(ascending=False)
    rows.append({"pair": f"{a} vs {b}", "default-rate gap": abs(rate[a] - rate[b]),
                 "largest feature gap": gap.index[0], "(in std units)": gap.iloc[0],
                 "2nd largest": gap.index[1], "(in std units) ": gap.iloc[1]})
display(pd.DataFrame(rows).set_index("pair").round(3))
print("cluster default rates:", rate.round(3).to_dict(), "| sizes:", sizes.tolist())

,default-rate gap,largest feature gap,(in std units),2nd largest,(in std units)
pair,,,,,
0 vs 1,0.427,late_months,2.277,delay_trend,1.428
0 vs 2,0.011,utilization,1.711,payment_ratio,1.600
1 vs 2,0.438,late_months,2.300,payment_ratio,1.640


cluster default rates: {0: 0.165, 1: 0.592, 2: 0.154} | sizes: [12449, 4293, 13258]


**Write-up (Part 3):** A small table of raw vs engineered default rates, then a direct answer: did engineering produce a sharper, more decision-useful risk separation? "Not much changed" and "it got worse" are legitimate findings **if your numbers show it**.

**Raw versus engineered, K = 3 (clusters ordered from riskiest to safest):**

| | Raw: customers | Raw: default rate | Engineered: customers | Engineered: default rate |
|---|---|---|---|---|
| Riskiest cluster | 15,924 | 0.266 | 4,293 | 0.592 |
| Middle cluster | 3,557 | 0.189 | 12,449 | 0.165 |
| Safest cluster | 10,519 | 0.165 | 13,258 | 0.154 |

For comparison, the lab went from 0.271 / 0.487 / 0.333 (raw) to 0.757 / 0.294 / 0.295 (engineered).

**Did engineering produce a sharper, more decision-useful separation? Yes.** The spread of default rates grew from 10.1 points to 43.8 points. The engineered riskiest cluster is 14.3% of customers with a 59.2% default rate (2.68 times the overall rate) and holds 38.3% of all defaulters. The raw riskiest cluster is 53.1% of customers with a 26.6% default rate (only 1.20 times the overall rate); it holds more defaulters (63.8%) only because it contains more than half of everyone. The pattern matches the lab (one sharp high-risk segment appears), though weaker: 59% here versus 76% there.

Two honest caveats. First, `late_months` comes from the same `PAY_*` columns that were already in my raw feature list, so engineering added no new information; it re-expressed it so K-Means could use it. Second, I chose the features from domain reasoning and never used the labels to build them. I kept `extreme_flag` out of K-Means and created no binned or category columns, so nothing 0/1 or categorical entered the distances.

**PCA.** PC1's share barely changed (32.7% raw, 32.9% engineered), unlike the lab, where PC1 shrank from 33.3% to about 20%. What changed is what PC1 represents. Raw PC1 was the bills axis (six `BILL_AMT` columns at about 0.35 each). Engineered PC1 is a credit-stress axis: `utilization` +0.597, `payment_ratio` -0.535, `log_limit` -0.393, `util_change` +0.365 (high use of the limit, paying little of what is owed, lower limit). Its correlation with default rose from 0.083 to 0.179. PC2 (17.7%) is lateness (`late_months` +0.657 against `util_change` -0.594), with a correlation of 0.290 (raw PC2: 0.286). The variance is also spread more evenly: PC3 explains 15.0% (7.5% raw), and 80% takes 5 components.

**Failure case (no winsorizing).** The cluster sizes became [8,914, 16,845, 4,241] instead of [12,449, 4,293, 13,258]. **No tiny cluster appeared.** The smallest cluster has 4,241 customers, and it is the high-risk lateness group (mean `late_months` 3.96 against 0.83 overall, default rate 0.591), essentially the same group as before (4,293 customers, 0.592). What changed is the boundary between the two low-risk clusters, which moved by about 3,500 customers. The 37 customers with a `payment_ratio` above 5 did not form their own cluster; they landed in all three ([6, 30, 1]). The visible damage is that the max `payment_ratio` is 376 instead of 1.33 and its standard deviation is 2.90 instead of 0.42, which squeezes everyone else's payment behavior closer together after scaling. My guess (not tested) is that this is what shifted the boundary, and that 37 customers out of 30,000 (0.12%) are too few to pull a cluster away the way the lab's handful of extreme debts did in 400 rows. So on this data, skipping winsorizing hurt less than in the lab.

**Do the clusters separate on risk or on something else?** Both, on different axes:

- **Clusters 0 vs 1 and 1 vs 2 separate on risk.** Their default-rate gaps are 0.427 and 0.438, and the biggest feature gap in both pairs is `late_months` (2.28 and 2.30 standard deviations).
- **Clusters 0 vs 2 separate on something else.** Their default rates differ by only 0.011 (0.165 vs 0.154), yet they differ by 1.71 standard deviations in `utilization` (0.73 vs 0.06) and 1.60 in `payment_ratio` (0.10 vs 0.77). One group carries big balances and pays little; the other uses almost none of its limit and pays most of what it owes. That is a difference in how they use credit, not in how likely they are to default.

Carrying a balance is not the same as being likely to default; persistent lateness is.